# Data Readiness Before Statistical Interpretation
This notebook only verifies and loads prepared inputs. No model calls, ledger access, hypothesis tests, or interpretation.
Primary: 896 attempts / 890 valid / 6 failures. Separate supplements: 9 attempts / 6 valid / 3 failures.
The combined 896 valid-graph catalog is descriptive only, not the primary inferential sample.

In [1]:
from pathlib import Path
import hashlib, json, sys
import importlib.metadata as package_metadata
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'outputs/analysis_ready/060dcd3a86d5/readiness.json').is_file())
READY = ROOT / 'outputs/analysis_ready/060dcd3a86d5'
receipt = json.loads((READY / 'readiness.json').read_text(encoding='utf8'))
assert receipt['status'] == 'READY_FOR_ANALYSIS_INPUT_REVIEW'
assert receipt['statistical_interpretation_performed'] is False
assert sys.version.split()[0] == receipt['runtime']['python']
for package, version in receipt['runtime'].items():
    assert package == 'python' or package_metadata.version(package) == version, package
for name, expected in receipt['generated_files'].items():
    data = (READY / name).read_bytes()
    assert len(data) == expected['bytes'] and hashlib.sha256(data).hexdigest() == expected['sha256'], name
print('Prepared-input hashes verified. Root:', ROOT)


Prepared-input hashes verified. Root: C:\Users\Hemu\Projects\UA_Captsone_SocN


## Load Separate Cohorts
Failures remain in the primary table. Undefined values stay missing; do not fill them with zero.
Use `primary` for primary-study work. `all_graphs_descriptive_only` includes selected extra observations and must not be passed to the primary hypothesis family.

In [2]:
primary = pd.read_csv(READY / 'primary_attempts.csv')
supplement = pd.read_csv(READY / 'supplemental_attempts.csv')
all_graphs_descriptive_only = pd.read_csv(READY / 'all_valid_graphs_descriptive_only.csv')
failures = pd.read_csv(READY / 'failures.csv')
assert len(primary) == 896 and int(primary['valid_graph'].sum()) == 890
assert primary['primary_member'].all() and primary['run_id'].is_unique
assert len(supplement) == 9 and int(supplement['valid_graph'].sum()) == 6
assert not supplement['primary_member'].any() and not supplement['eligible_primary_graph'].any()
assert len(failures) == 9 and len(all_graphs_descriptive_only) == 896
print(pd.DataFrame({'attempts':[len(primary),len(supplement)], 'valid_graphs':[int(primary.valid_graph.sum()),int(supplement.valid_graph.sum())]}, index=['primary','supplement']))


            attempts  valid_graphs
primary          896           890
supplement         9             6


In [3]:
for row in all_graphs_descriptive_only.itertuples(index=False):
    path = ROOT / row.adj_path
    assert path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest() == row.adj_sha256, row.run_id
assert failures['record_path'].map(lambda p: (ROOT / p).is_file()).all()
primary_analysis_path = ROOT / 'outputs/final_study/060dcd3a86d5/main/analysis_rows.json'
print('896 adjacency paths/hashes and 9 failure paths verified.')
print('Existing primary-analysis rows remain at:', primary_analysis_path)


896 adjacency paths/hashes and 9 failure paths verified.
Existing primary-analysis rows remain at: C:\Users\Hemu\Projects\UA_Captsone_SocN\outputs\final_study\060dcd3a86d5\main\analysis_rows.json


## Next Stage, Not Run Here
Review endpoint definitions, missingness, complete/partial schedule blocks, inference assumptions, and method compute imbalance before interpreting existing contrasts.
Original analysis outputs remain untouched. No p-values, effect claims, sensitivity conclusions, or pooled primary analysis are produced by this notebook.